# YouTube Direct Uploader
Uploads a video from your Colab filesystem directly to your YouTube channel.
All permissions are requested upfront in Step 1.

In [ ]:
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

CLIENT_ID     = 'YOUR_CLIENT_ID'
CLIENT_SECRET = 'YOUR_CLIENT_SECRET'
REFRESH_TOKEN = 'YOUR_REFRESH_TOKEN'

creds = Credentials(
    token=None,
    refresh_token=REFRESH_TOKEN,
    token_uri='https://oauth2.googleapis.com/token',
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scopes=['https://www.googleapis.com/auth/youtube']
)
creds.refresh(Request())
print('Authenticated.')

In [14]:
# Step 2: Configure video details
# ---- CONFIGURE THESE ----
LOCAL_VIDEO_PATH = '/content/video.mp4'  # path to your video in Colab or Drive
TITLE            = 'My Video Title'
DESCRIPTION      = 'Uploaded from Colab'
PRIVACY          = 'public'              # public / unlisted / private
TAGS             = ['colab', 'python']
# -------------------------
import os
print(f'Video: {LOCAL_VIDEO_PATH}')
print(f'Size:  {os.path.getsize(LOCAL_VIDEO_PATH)/1_048_576:.2f} MB')

Video: /content/video.mp4
Size:  0.75 MB


In [15]:
LOCAL_VIDEO_PATH = '/content/video.mp4'
TITLE            = 'My Video Title'
DESCRIPTION      = 'Uploaded from Colab'
PRIVACY          = 'public'
TAGS             = ['colab', 'python']

In [16]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

youtube = build('youtube', 'v3', credentials=creds)
media = MediaFileUpload(LOCAL_VIDEO_PATH, mimetype='video/mp4', resumable=True)
req = youtube.videos().insert(
    part='snippet,status',
    body={
        'snippet': {'title': TITLE, 'description': DESCRIPTION, 'tags': TAGS},
        'status':  {'privacyStatus': PRIVACY}
    },
    media_body=media
)

response = None
while response is None:
    status, response = req.next_chunk()
    if status:
        print(f'{int(status.progress() * 100)}%')

print(f'https://youtu.be/{response["id"]}')

https://youtu.be/c0kw1N3872M
